<a href="https://colab.research.google.com/github/greeshmanthvarma/Cybersecurity_LLM_Finetuning/blob/main/gemma_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q -U transformers bitsandbytes accelerate trl peft
!pip install -q -U rouge-score bert-score datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

In [ ]:
import torch
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from rouge_score import rouge_scorer
from bert_score import score as bert_score_fn
from huggingface_hub import login

login()

MODEL_ID = "google/gemma-4-E4B-it"
SEED = 42

print(torch.cuda.get_device_name(0))
print(f"Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


In [ ]:
print("Loading dataset...")
ds = load_dataset("Trendyol/Trendyol-Cybersecurity-Instruction-Tuning-Dataset")

split1 = ds["train"].train_test_split(test_size=0.1, seed=SEED)
split2 = split1["test"].train_test_split(test_size=0.5, seed=SEED)

train_data = split1["train"]
val_data   = split2["train"]
test_data  = split2["test"]

print(f"Train: {len(train_data)}")
print(f"Val:   {len(val_data)}")
print(f"Test:  {len(test_data)}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_example(example):
    messages = [
        {"role": "system",    "content": example["system"]},
        {"role": "user",      "content": example["user"]},
        {"role": "assistant", "content": example["assistant"]},
    ]
    return {"text": tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )}


In [ ]:
train_data_formatted = train_data.map(format_example)
val_data_formatted   = val_data.map(format_example)

def count_tokens(example):
    tokens = tokenizer(example["text"], return_tensors="pt")
    return {"token_count": tokens["input_ids"].shape[1]}

train_data_with_counts = train_data_formatted.map(count_tokens)
counts = train_data_with_counts["token_count"]

print(f"Min:             {np.min(counts)}")
print(f"Max:             {np.max(counts)}")
print(f"Mean:            {np.mean(counts):.0f}")
print(f"Median:          {np.median(counts):.0f}")
print(f"95th percentile: {np.percentile(counts, 95):.0f}")
print(f"Examples over 4096: {sum(1 for c in counts if c > 4096)}")

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)
model = prepare_model_for_kbit_training(model)

print(f"VRAM after model load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules="all-linear",
)

In [ ]:
%%javascript
function ClickConnect(){
    console.log("Keeping alive...");
    document.querySelector("colab-connect-button").click()
}
setInterval(ClickConnect, 60000)

In [ ]:
training_args = SFTConfig(
    output_dir="/content/drive/MyDrive/cybersec-finetuned",
    num_train_epochs=1,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    logging_steps=25,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    load_best_model_at_end=True,
    learning_rate=2e-4,
    bf16=True,
    gradient_checkpointing=True,
    dataset_text_field="text",
)

tokenizer.model_max_length = 1536

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data_formatted,
    eval_dataset=val_data_formatted,
    processing_class=tokenizer,
    peft_config=peft_config,
)

print(f"VRAM before training: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
trainer.train(resume_from_checkpoint="/content/drive/MyDrive/cybersec-finetuned/checkpoint-1400")

# Save final model
trainer.save_model("/content/drive/MyDrive/cybersec-finetuned")
print("Training complete. Model saved.")

In [ ]:
MAX_SAMPLES    = 50
MAX_NEW_TOKENS = 128

def build_prompt(example):
    messages = [
        {"role": "system", "content": example["system"]},
        {"role": "user",   "content": example["user"]},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

rows       = test_data.select(range(min(MAX_SAMPLES, len(test_data))))
references = [r["assistant"].strip() for r in rows]

print(f"\nLoading base {MODEL_ID} for inference...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
base_model.eval()
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

base_predictions = []
for i, ex in enumerate(rows):
    prompt = build_prompt(ex)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(base_model.device) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        out = base_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    pred = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()
    base_predictions.append(pred)
    if (i + 1) % 10 == 0 or i == 0:
        print(f"[{i+1}/{len(rows)}] input: {ex['user'][:60]}...")
        print(f"           output: {pred[:60]}...")

del base_model
torch.cuda.empty_cache()

In [ ]:
!pip install -q torchao --upgrade
import importlib
import peft
importlib.reload(peft)
from peft import PeftModel

print(f"\nLoading fine-tuned model for inference...")
ft_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
ft_model = PeftModel.from_pretrained(ft_base, "/content/drive/MyDrive/cybersec-finetuned")
ft_model.eval()
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

ft_predictions = []
for i, ex in enumerate(rows):
    prompt = build_prompt(ex)
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(ft_model.device) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        out = ft_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    pred = tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()
    ft_predictions.append(pred)
    if (i + 1) % 10 == 0 or i == 0:
        print(f"[{i+1}/{len(rows)}] input: {ex['user'][:60]}...")
        print(f"           output: {pred[:60]}...")

del ft_model
torch.cuda.empty_cache()

In [ ]:
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

base_rouge = [rouge.score(ref, pred)["rougeL"].fmeasure
              for ref, pred in zip(references, base_predictions)]
ft_rouge   = [rouge.score(ref, pred)["rougeL"].fmeasure
              for ref, pred in zip(references, ft_predictions)]

print(f"ROUGE-L  | Base: {sum(base_rouge)/len(base_rouge):.4f} | Fine-tuned: {sum(ft_rouge)/len(ft_rouge):.4f}")

_, _, base_bert = bert_score_fn(base_predictions, references,
                                lang="en", model_type="roberta-large",
                                device="cpu", use_fast_tokenizer=False, verbose=False)
_, _, ft_bert   = bert_score_fn(ft_predictions, references,
                                lang="en", model_type="roberta-large",
                                device="cpu", use_fast_tokenizer=False, verbose=False)

print(f"BERTScore| Base: {base_bert.mean().item():.4f} | Fine-tuned: {ft_bert.mean().item():.4f}")

In [ ]:
!pip install openai

from openai import OpenAI
import json
import time
from google.colab import userdata

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=OPENAI_API_KEY)

JUDGE_PROMPT = """You are a cybersecurity expert evaluating the quality of AI-generated responses to security questions.

Rate the following response on a scale of 1 to 5 using these criteria:
- 5: Technically accurate, complete, well-structured, and directly addresses the question
- 4: Mostly accurate and complete with minor gaps
- 3: Partially correct but missing important details or contains minor inaccuracies
- 2: Superficial or significantly incomplete with notable inaccuracies
- 1: Incorrect, irrelevant, or dangerously misleading

Return ONLY a JSON object with no preamble or markdown. Example:
{{"score": 4, "reason": "Response correctly identifies the attack vector but omits remediation steps."}}

Question: {question}
Response: {response}"""

def judge_response(question, response, retries=3):
    for attempt in range(retries):
        try:
            completion = client.chat.completions.create(
                model="gpt-4o-mini",
                max_tokens=200,
                messages=[
                    {"role": "user", "content": JUDGE_PROMPT.format(
                        question=question,
                        response=response[:2000]
                    )}
                ]
            )
            raw = completion.choices[0].message.content.strip()
            return json.loads(raw)
        except json.JSONDecodeError:
            print(f"  JSON parse failed on attempt {attempt+1}, retrying...")
            time.sleep(1)
        except Exception as e:
            print(f"  API error on attempt {attempt+1}: {e}")
            time.sleep(2)
    return {"score": None, "reason": "failed after retries"}


def evaluate_with_judge(predictions, rows, label=""):
    print(f"\nRunning LLM judge on {label} predictions...")
    results = []
    for i, (pred, ex) in enumerate(zip(predictions, rows)):
        result = judge_response(ex["user"], pred)
        results.append(result)
        if (i + 1) % 10 == 0 or i == 0:
            print(f"  [{i+1}/{len(predictions)}] score: {result.get('score')} | {result.get('reason', '')[:80]}")
        time.sleep(0.3)

    valid_scores = [r["score"] for r in results if r["score"] is not None]
    mean_score = sum(valid_scores) / len(valid_scores) if valid_scores else 0
    print(f"\n{label} Judge Score (mean): {mean_score:.3f} (n={len(valid_scores)})")
    return results, mean_score


base_judge_results, base_judge_mean = evaluate_with_judge(base_predictions, rows, label="Base Gemma E4B")
ft_judge_results,   ft_judge_mean   = evaluate_with_judge(ft_predictions,   rows, label="Fine-tuned Gemma E4B")

In [ ]:
FOUNDATION_MODEL_ID = "fdtn-ai/Foundation-Sec-8B"

print(f"\nLoading {FOUNDATION_MODEL_ID}...")
foundation_tokenizer = AutoTokenizer.from_pretrained(FOUNDATION_MODEL_ID)
if foundation_tokenizer.pad_token is None:
    foundation_tokenizer.pad_token = foundation_tokenizer.eos_token

foundation_model = AutoModelForCausalLM.from_pretrained(
    FOUNDATION_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
)
foundation_model.eval()
print(f"VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

def build_foundation_prompt(example):
    return (
        f"### System:\n{example['system']}\n\n"
        f"### User:\n{example['user']}\n\n"
        f"### Assistant:\n"
    )

foundation_predictions = []
for i, ex in enumerate(rows):
    prompt = build_foundation_prompt(ex)
    inputs = foundation_tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(foundation_model.device) for k, v in inputs.items()}
    prompt_len = inputs["input_ids"].shape[1]
    with torch.inference_mode():
        out = foundation_model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=foundation_tokenizer.pad_token_id,
            eos_token_id=foundation_tokenizer.eos_token_id,
        )
    pred = foundation_tokenizer.decode(out[0, prompt_len:], skip_special_tokens=True).strip()
    foundation_predictions.append(pred)
    if (i + 1) % 10 == 0 or i == 0:
        print(f"[{i+1}/{len(rows)}] input : {ex['user'][:60]}...")
        print(f"           output: {pred[:60]}...")

del foundation_model
torch.cuda.empty_cache()

# Judge evaluation for Foundation-Sec-8B
foundation_judge_results, foundation_judge_mean = evaluate_with_judge(
    foundation_predictions, rows, label="Foundation-Sec-8B"
)

In [ ]:
foundation_rouge = [rouge.score(ref, pred)["rougeL"].fmeasure
                    for ref, pred in zip(references, foundation_predictions)]

_, _, foundation_bert = bert_score_fn(
    foundation_predictions, references,
    lang="en", model_type="roberta-large",
    device="cpu", use_fast_tokenizer=False, verbose=False
)

print("\n" + "="*65)
print(f"{'Metric':<20} {'Base E4B':>12} {'Fine-tuned E4B':>15} {'Foundation-8B':>14}")
print("="*65)
print(f"{'ROUGE-L F1':<20} {sum(base_rouge)/len(base_rouge):>12.4f} {sum(ft_rouge)/len(ft_rouge):>15.4f} {sum(foundation_rouge)/len(foundation_rouge):>14.4f}")
print(f"{'BERTScore F1':<20} {base_bert.mean().item():>12.4f} {ft_bert.mean().item():>15.4f} {foundation_bert.mean().item():>14.4f}")
print(f"{'LLM Judge (1-5)':<20} {base_judge_mean:>12.3f} {ft_judge_mean:>15.3f} {foundation_judge_mean:>14.3f}")
print("="*65)

In [ ]:
import matplotlib.pyplot as plt

steps = [200,600,1000,1400,1800,2200,2600,2993]
train_loss = [0.7067,0.6270,0.6098,0.5657,0.5622,0.5585,0.5420,0.5533]
val_loss = [0.7166,0.6394,0.6021,0.5825,0.5686,0.5570,0.5485,0.5440]

plt.figure(figsize=(10,5))
plt.plot(steps, train_loss, label="Training Loss")
plt.plot(steps, val_loss, label="Validation Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Gemma 4 E4B QLoRA Fine-tuning — Loss Curves")
plt.legend()
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=150)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

metrics = ["ROUGE-L F1", "BERTScore F1", "LLM Judge (1-5)"]
base = [0.1284, 0.8284, 3.900]
finetuned = [0.1857, 0.8628, 4.180]
foundation = [0.1481, 0.8341, 3.780]

x = np.arange(len(metrics))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x - width, base, width, label="Base Gemma E4B")
ax.bar(x, finetuned, width, label="Fine-tuned Gemma E4B")
ax.bar(x + width, foundation, width, label="Foundation-Sec-8B")

ax.set_xticks(x)
ax.set_xticklabels(metrics)
ax.set_title("Evaluation Results: Base vs Fine-tuned Gemma E4B vs Foundation-Sec-8B")
ax.legend()
ax.set_ylim(0, 5.5)
plt.tight_layout()
plt.savefig("eval_metrics.png", dpi=150)
plt.show()